# 29. The plasticity probe, Tiny ImageNet only

This runs the adapter plasticity study's probe on this framework, using nothing but Tiny
ImageNet. It replaces the approximation in notebook 26.

### What a probe is

Take a copy of the model. Train the copy on a task it has never seen. Record how well it
learns. Throw the copy away. Because the copy is discarded, measuring never changes the run
being measured. The study calls this fork and discard.

### The two endpoints

| Name | What it trains on | What it reports | Plasticity loss shows as |
| --- | --- | --- | --- |
| **E1** | images with randomly shuffled labels | final training loss | a rise |
| **E2** | three tasks the model has never seen | test accuracy | a drop |

E1 measures raw capacity to change. Random labels carry no pattern, so the only way to fit
them is to memorise. E2 measures useful learning.

### How the parts map

That study trains a small adapter on a frozen pretrained backbone, plus a fresh head per
probe task. Here the **generator** plays the adapter's part, because it is the one network
every task updates, and a **fresh 32-number task code** plays the head's part.

- **Full probe**: generator and fresh code both train.
- **Code-only probe**: generator frozen, only the fresh code trains. This is that study's
  head-only control.

### Probe points

Two: **t = 0**, a generator that has learned nothing, and **t = 8**, after the eight tasks
of order 01. Both already exist, so no stream has to be re-run. Mid-stream points would
need that, and are left for later.

### Why this is better than notebook 26

| | Notebook 26 | Here |
| --- | --- | --- |
| Probe tasks | 1 | 3, plus a random-label task |
| Learning rate | one fixed | best of a grid |
| Repetitions | 2 per seed | 5 at t = 0, 2 at t = 8 |
| Protocol | trained fresh models | fork and discard, verified |
| Noise floor | borrowed, 3 points | measured from the repetitions |
| Chunk embeddings | trained at t = 0, not at t = 8 | never trained, at either point |

That last row removes the confound notebook 26 had to work around. The probe never trains
the chunk embeddings, so t = 0 and t = 8 are on equal footing.

### What Tiny ImageNet cannot give

Of that study's three natural probe tasks only EuroSAT, at ten classes, could ever fit
here. DTD has 47 classes and Flowers102 has 102, and the generator's output layout is fixed
at ten. More importantly, probe tasks drawn from the same dataset as the stream do not
separate plasticity from transfer between similar classes. That study uses unrelated tasks
on purpose. Any result here must say so.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.probe import (RandomLabels, best_of_grid, noise_floor, probe_once,
                         relative_contribution, snapshot, unchanged)

sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError as problem:
    print('Progress panel unavailable, using plain bars:', problem)

    class Dashboard:
        """Just enough of the panel to keep the notebook running."""

        def __init__(self, *args, **kwargs): pass
        def __enter__(self): return self
        def __exit__(self, *args): return False
        def note(self, text): print(text, flush=True)

        def bar(self, total, description, unit='step'):
            from uncle.telemetry import progress_bar
            return progress_bar(total, description)

import torch
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)

## 2. Settings

The defaults follow the adapter study where this framework allows it. The cell prints how
many probe runs that implies and roughly how long they take, so the budget is visible
before anything starts.

To shorten a first pass, drop `PROBE_TASKS` to one task or `RATES` to one rate. Do not
change `STEPS` between probe points: that would measure the budget instead of the model.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"
SEED = 0                      # 0, 1 or 2. Run each separately.

# Tasks the stream never saw. Order 01 learns 3, 0, 9, 5, 17, 1, 7 and 14, so
# twelve of the twenty remain. These three are the controls the conclusion
# report already used.
PROBE_TASKS = ["2", "4", "6"]

# Images for the random-label probe, taken from another unseen task so its
# images are not reused by E2.
RANDOM_LABEL_TASK = "8"
RANDOM_LABEL_SEED = 1020      # fixes the wrong labels, shared by every point

# Memorising random labels needs many passes over few images. The first run of
# this notebook used the task's full 5,000 images for 400 steps, which is about
# 5 passes, and the loss never fell below chance: the probe measured nothing.
# These are the adapter study's own dose settings, chosen after it hit the same
# problem: 2,000 images for 600 steps, about 19 passes.
RANDOM_LABEL_IMAGES = 2000

# A model late in a stream may need a different rate than a fresh one, so the
# probe tries a grid and keeps the best. A single fixed rate would confuse a
# rate mismatch with plasticity loss.
RATES = [1e-4, 3e-4, 1e-3]

STEPS = 250                   # natural probe tasks, as the study uses
RANDOM_LABEL_STEPS = 600      # about 19 passes over 2,000 images
BATCH_SIZE = 64
FEATURE_IMAGES = 2000         # for the effective rank

# More repetitions at t = 0, because its spread is what the noise floor is
# built from. The study uses 5 and 2.
REPETITIONS_AT_ZERO = 5
REPETITIONS_LATER = 2

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"

runs_at = lambda reps: (len(PROBE_TASKS) + 1) * len(RATES) * reps * 2
TOTAL_RUNS = runs_at(REPETITIONS_AT_ZERO) + runs_at(REPETITIONS_LATER)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(f"Probe runs: {TOTAL_RUNS}  "
      f"({len(PROBE_TASKS)} natural tasks + 1 random-label, {len(RATES)} rates, "
      f"2 arms, {REPETITIONS_AT_ZERO} and {REPETITIONS_LATER} repetitions)")
print(f"At roughly 0.6 minutes each that is about {TOTAL_RUNS * 0.6 / 60:.1f} hours.")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the images and the two models

`t = 8` is the finished source model for this seed. `t = 0` is a generator built with the
same settings and the same seed, which has learned nothing.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]

overlap = set(PROBE_TASKS + [RANDOM_LABEL_TASK]) & set(ORDER_TASKS)
if overlap:
    raise ValueError(f"These probe tasks are in the learning order: {sorted(overlap)}")

needed = tuple(ORDER_TASKS + PROBE_TASKS + [RANDOM_LABEL_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded images for", len(needed), "tasks")

source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

def build(restore_source):
    """A model at one probe point. restore_source False gives the t = 0 model."""
    config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                       "requests": tuple(("learn", t) for t in ORDER_TASKS)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if restore_source:
        checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    return target, hypernet, trainer

# The random labels are drawn once and shared by every probe point. A fresh
# draw per point would make the points incomparable.
from torch.utils.data import Subset
# Fewer images, so the fixed step budget gives enough passes to memorise them.
# The subset is drawn once with a fixed seed and shared by every probe point.
pool = tasks[RANDOM_LABEL_TASK]["train"]
chosen = torch.randperm(len(pool),
                        generator=torch.Generator().manual_seed(RANDOM_LABEL_SEED)
                        )[:RANDOM_LABEL_IMAGES].tolist()
random_label_set = RandomLabels(Subset(pool, chosen), CLASSES, RANDOM_LABEL_SEED)
print(f"Random-label probe: {len(random_label_set)} images, {RANDOM_LABEL_STEPS} steps, "
      f"about {RANDOM_LABEL_STEPS * BATCH_SIZE / len(random_label_set):.0f} passes")
print("Probe tasks:", PROBE_TASKS, "| random labels from task", RANDOM_LABEL_TASK)

## 5. Run the grid

Every probe run snapshots the model, trains a fresh code on a probe task, measures, then
puts the model back. The cell checks after each run that the model really is unchanged, and
stops if it is not.

In [ ]:
import time

results = []

def grid_at(point, restore_source, repetitions):
    """Every probe task, rate, arm and repetition at one probe point."""
    target, hypernet, trainer = build(restore_source)
    reference = snapshot(hypernet, trainer)
    jobs_here = [(task, tasks[task]["train"], tasks[task]["test"], STEPS, "E2")
                 for task in PROBE_TASKS]
    jobs_here.append((RANDOM_LABEL_TASK, random_label_set, None,
                      RANDOM_LABEL_STEPS, "E1"))
    total = len(jobs_here) * len(RATES) * repetitions * 2
    with Dashboard(f"Probe at t = {point}, seed {SEED}") as board:
        bar = board.bar(total, f"t={point}", unit="run")
        for task, train_set, test_set, steps, endpoint in jobs_here:
            for rate in RATES:
                for code_only in (False, True):
                    for repetition in range(repetitions):
                        board.note(f"{endpoint} task {task} lr {rate:g} "
                                   f"{'code-only' if code_only else 'full'} "
                                   f"rep {repetition + 1}")
                        run = probe_once(
                            hypernet, target, trainer, task=task,
                            train_set=train_set, test_set=test_set, steps=steps,
                            learning_rate=rate, batch_size=BATCH_SIZE,
                            code_only=code_only,
                            # The seed depends on the probe task and the
                            # repetition, never on the probe point, so the
                            # points differ only by the model's state.
                            seed=12345 + 1000 * repetition + int(task),
                            feature_images=FEATURE_IMAGES if endpoint == "E2" else 0)
                        if not unchanged(hypernet, reference):
                            raise RuntimeError("A probe changed the model. Stop and fix "
                                               "this before trusting anything.")
                        run.update(point=point, endpoint=endpoint, repetition=repetition,
                                   seed_label=SEED)
                        results.append(run)
                        bar.update(1)
        bar.close()
    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

began = time.perf_counter()
grid_at(0, restore_source=False, repetitions=REPETITIONS_AT_ZERO)
grid_at(8, restore_source=True, repetitions=REPETITIONS_LATER)
ELAPSED_MIN = (time.perf_counter() - began) / 60
print(f"\n{len(results)} probe runs in {ELAPSED_MIN:.1f} minutes. "
      "The model was verified unchanged after every one.")

## 6. Pick the best rate, then average

For each probe point, task, arm and repetition, the grid keeps one run: the highest test
accuracy for E2, the lowest final training loss for E1. Then the repetitions are averaged,
and for E2 the three tasks are averaged.

In [ ]:
import statistics

def selected(point, endpoint, task, code_only, repetition):
    """The best rate for one probe point, task, arm and repetition."""
    runs = [r for r in results if r["point"] == point and r["endpoint"] == endpoint
            and r["task"] == task and r["code_only"] == code_only
            and r["repetition"] == repetition]
    return best_of_grid(runs, endpoint=endpoint) if runs else None

def value_of(run, endpoint):
    return run["test_accuracy"] if endpoint == "E2" else run["final_train_loss"]

def endpoint_values(point, endpoint, code_only):
    """One number per repetition: averaged over probe tasks for E2."""
    which = PROBE_TASKS if endpoint == "E2" else [RANDOM_LABEL_TASK]
    reps = REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER
    out = []
    for repetition in range(reps):
        picked = [selected(point, endpoint, task, code_only, repetition) for task in which]
        if any(p is None for p in picked):
            continue
        out.append(sum(value_of(p, endpoint) for p in picked) / len(picked))
    return out

summary = {}
for endpoint in ("E2", "E1"):
    for code_only in (False, True):
        for point in (0, 8):
            values = endpoint_values(point, endpoint, code_only)
            summary[(endpoint, code_only, point)] = values

WIDTH = 92
def rule(title):
    print(f"\n{title}\n" + "-" * WIDTH)

print("=" * WIDTH)
print(f"PLASTICITY PROBE, seed {SEED}, order {ORDER}".center(WIDTH))
print(f"{len(results)} runs | {ELAPSED_MIN:.1f} min | Tiny ImageNet probe tasks".center(WIDTH))
print("=" * WIDTH)

rule("TABLE 1. Endpoint values after choosing the best rate, per repetition")
print(f"{'endpoint':<10}{'arm':<12}{'point':>7}{'reps':>6}{'mean':>10}{'sd':>9}  values")
print("-" * WIDTH)
for (endpoint, code_only, point), values in summary.items():
    if not values:
        continue
    arm = "code-only" if code_only else "full"
    sd = statistics.stdev(values) if len(values) > 1 else float("nan")
    shown = " ".join(f"{v:.2f}" for v in values)
    print(f"{endpoint:<10}{arm:<12}{point:>7}{len(values):>6}"
          f"{sum(values) / len(values):>10.2f}{sd:>9.2f}  {shown}")
print("-" * WIDTH)
print("E2 is test accuracy in percent, averaged over the three probe tasks. Higher is")
print("better. E1 is final training loss on random labels. Lower is better. Chance loss")
print(f"for {CLASSES} classes is ln({CLASSES}) = {__import__('math').log(CLASSES):.3f}.")

## 7. The decision rule

The adapter study declares plasticity loss only when **both** hold:

1. the change from t = 0 has the right sign and its interval excludes zero, and
2. the change is larger than the noise floor.

The noise floor comes from the spread of repetitions at t = 0:

    d = 1.96 * s * sqrt(1/n_t + 1/n_0)

This notebook can finally check both, because it repeats the probe five times at t = 0.

In [ ]:
import math

CHANCE_LOSS = math.log(CLASSES)
# The random-label probe only measures anything once it has actually memorised
# some labels. A final loss sitting at chance means the budget was too small,
# not that capacity is unchanged, and the decision rule must not be applied to
# it. The study requires the probe to fit "well below chance loss".
E1_WORKED = all(
    summary[(key, False, point)] and
    sum(summary[(key, False, point)]) / len(summary[(key, False, point)])
    < CHANCE_LOSS - 0.20
    for key, point in (("E1", 0), ("E1", 8)))
if not E1_WORKED:
    print(f"WARNING. The random-label probe did not memorise. Chance loss is "
          f"{CHANCE_LOSS:.3f} and the full arm ends near it.")
    print("E1 is reported below but means nothing: it measures the budget, not the")
    print("model. Raise RANDOM_LABEL_STEPS or lower RANDOM_LABEL_IMAGES and rerun.")
    print()

rule("TABLE 2. The decision rule, full arm")
print(f"{'endpoint':<10}{'t=0':>9}{'t=8':>9}{'change':>10}{'floor':>9}"
      f"{'exceeds':>9}  verdict")
print("-" * WIDTH)

verdicts = {}
for endpoint in ("E2", "E1"):
    zero = summary[(endpoint, False, 0)]
    later = summary[(endpoint, False, 8)]
    if len(zero) < 2 or not later:
        print(f"{endpoint:<10}  not enough repetitions to decide")
        continue
    mean_zero, mean_later = sum(zero) / len(zero), sum(later) / len(later)
    change = mean_later - mean_zero
    floor = noise_floor(statistics.stdev(zero), statistics.stdev(zero),
                        len(zero), len(later))
    exceeds = abs(change) > floor
    # E2 is accuracy, so a fall is loss. E1 is loss, so a rise is loss.
    loses = change < 0 if endpoint == "E2" else change > 0
    if endpoint == "E1" and not E1_WORKED:
        verdict = "VOID, the probe never memorised"
    else:
        verdict = ("PLASTICITY LOSS" if exceeds and loses else
                   "no loss, change is the other way" if exceeds else
                   "NO FINDING, inside the noise floor")
    verdicts[endpoint] = {"t0": mean_zero, "t8": mean_later, "change": change,
                          "probe_valid": E1_WORKED if endpoint == "E1" else True,
                          "noise_floor": floor, "exceeds_floor": exceeds,
                          "direction_is_loss": loses, "verdict": verdict}
    print(f"{endpoint:<10}{mean_zero:>9.2f}{mean_later:>9.2f}{change:>+10.2f}"
          f"{floor:>9.2f}{('yes' if exceeds else 'no'):>9}  {verdict}")
print("-" * WIDTH)
print("The floor is built from the spread of the five repetitions at t = 0, which is")
print("this measurement's own wobble rather than a number borrowed from elsewhere.")

rule("TABLE 3. The code-only control, and what it says about R(t)")
for endpoint in ("E2", "E1"):
    for point in (0, 8):
        full = summary[(endpoint, False, point)]
        control = summary[(endpoint, True, point)]
        if not full or not control:
            continue
        gap = (sum(full) / len(full)) - (sum(control) / len(control))
        if endpoint == "E1":
            gap = -gap          # for a loss, the control minus the full is the gain
        print(f"  {endpoint} at t={point}: full {sum(full) / len(full):.2f}, "
              f"code-only {sum(control) / len(control):.2f}, "
              f"generator contributes {gap:+.2f}")
print()
for endpoint in ("E2", "E1"):
    full0, ctrl0 = summary[(endpoint, False, 0)], summary[(endpoint, True, 0)]
    full8, ctrl8 = summary[(endpoint, False, 8)], summary[(endpoint, True, 8)]
    if not all((full0, ctrl0, full8, ctrl8)):
        continue
    mean = lambda v: sum(v) / len(v)
    try:
        ratio = relative_contribution(mean(full8), mean(ctrl8), mean(full0), mean(ctrl0),
                                      endpoint=endpoint)
        note = "" if endpoint == "E2" or E1_WORKED else "   (void, the probe never memorised)"
        print(f"  R(8) for {endpoint} = {ratio:.3f}{note}")
    except ValueError as problem:
        print(f"  R(8) for {endpoint}: not computable. {problem}")
print()
print("R(t) divides the contribution at t = 8 by the contribution at t = 0. Read the")
print("t = 0 line first. In the adapter study the control there is a linear probe on")
print("pretrained features and scores well, so the denominator is solid. Here the")
print("control at t = 0 is a frozen random generator with a 32-number code. If its")
print("contribution is near zero, R(t) divides by almost nothing and means little.")
print("Report the contributions themselves rather than the ratio when that happens.")

rule("TABLE 4. Feature effective rank, full arm, E2 tasks")
print(f"{'point':>7}{'mean rank':>12}{'sd':>9}{'runs':>8}")
print("-" * WIDTH)
for point in (0, 8):
    reps = REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER
    # Only the runs the grid chose. Averaging over every rate mixes in runs at
    # a rate that did not suit the model, which is most of the spread.
    ranks = [selected(point, "E2", task, False, repetition)["effective_rank"]
             for task in PROBE_TASKS for repetition in range(reps)
             if selected(point, "E2", task, False, repetition)
             and "effective_rank" in selected(point, "E2", task, False, repetition)]
    if ranks:
        sd = statistics.stdev(ranks) if len(ranks) > 1 else float("nan")
        print(f"{point:>7}{sum(ranks) / len(ranks):>12.1f}{sd:>9.1f}{len(ranks):>8}")
print("-" * WIDTH)
print("Effective rank says how many directions the features really use, out of the")
print("layer's full width. A fall over the stream is a known plasticity warning sign.")
print("Probe tasks differ from each other, so a spread of this size across tasks is")
print("expected. Compare the two points, not one number against the layer width.")

## 8. Save

In [ ]:
import csv, datetime, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT = CACHE / "diagnostics" / f"plasticity_probe_{ORDER}_seed{SEED}_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "29_plasticity_probe",
    "protocol": ("Fork and discard, as in reference/adapter-plasticity-study. The model is "
                 "snapshotted, a fresh task code is trained on an unseen task, the result "
                 "is measured, and the model is restored and verified unchanged."),
    "limitations": [
        "Probe tasks come from the same dataset as the stream, so plasticity and "
        "transfer between similar classes are not separated.",
        "Of the adapter study's three natural probe tasks only EuroSAT could fit, "
        "because the generator's output layout is fixed at ten classes.",
        "Two probe points only, t = 0 and t = 8. Mid-stream points need the stream "
        "re-run with probing inline.",
        "One seed per run of this notebook.",
    ],
    "code_revision": HEAD, "study_id": STUDY_ID, "order": ORDER, "seed": SEED,
    "probe_tasks": PROBE_TASKS, "random_label_task": RANDOM_LABEL_TASK,
    "rates": RATES, "steps": STEPS, "random_label_steps": RANDOM_LABEL_STEPS,
    "batch_size": BATCH_SIZE, "repetitions": {"0": REPETITIONS_AT_ZERO, "8": REPETITIONS_LATER},
    "elapsed_min": ELAPSED_MIN,
    "environment": {"device": DEVICE,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
                    "torch": torch.__version__, "python": sys.version.split()[0],
                    "platform": platform.platform()},
    "verdicts": verdicts,
    "summary": {f"{e}|{'code_only' if c else 'full'}|t{p}": v
                for (e, c, p), v in summary.items()},
    "runs": results,
}
(EXPORT / "probe.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")

with (EXPORT / "runs.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["point", "endpoint", "task", "learning_rate", "code_only", "repetition",
               "steps", "final_train_loss", "final_train_accuracy", "test_accuracy",
               "test_loss", "effective_rank"]
    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)

SAVE_VERIFIED = True
for name in ("probe.json", "runs.csv"):
    path = EXPORT / name
    try:
        text = path.read_text(encoding="utf-8")
        if name.endswith(".json"):
            json.loads(text)
        elif len(text.splitlines()) != len(results) + 1:
            raise ValueError("row count does not match the runs")
        print(f"  verified  {name:<14} {path.stat().st_size:>9,} bytes")
    except Exception as problem:
        SAVE_VERIFIED = False
        print(f"  FAILED    {name:<14} {problem}")

SAVED_LOCATION = str(EXPORT)
print("\nSaved locally:", EXPORT)
if SAVE_TO_R2 and SAVE_VERIFIED:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    for name in ("probe.json", "runs.csv"):
        store.upload(EXPORT / name, f"{destination}/{name}")
    SAVED_LOCATION = destination
    print("Uploaded to R2:", destination)
elif SAVE_TO_R2:
    print("Not uploading: the local files did not verify.")

## 9. Release the GPU

Only once every file has been written and read back. If anything failed to save the
runtime stays connected, so hours of probing are not lost.

In [ ]:
if not SAVE_VERIFIED:
    print("NOT releasing the GPU: some output files are missing or unreadable.")
    print("Rerun section 8 before closing this runtime, or the probe runs are lost.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Share that path. Run this notebook again with SEED 1 and 2.")
    import gc
    for name in ("state", "tasks", "random_label_set"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## 10. What this leaves

Run this for seeds 0, 1 and 2, then the Tiny ImageNet part of the adapter study's
measurement set is done except for three things, and two of those are blocked by this
framework rather than by the data.

| Metric | After this notebook |
| --- | --- |
| Final accuracy, learning accuracy, forgetting, backward transfer, curve slope | Done, notebook 28 |
| E1, random-label loss | Done here |
| E2, natural-task accuracy | Done here, with same-dataset probe tasks |
| Noise floor, and the decision rule | Done here |
| Best-of-grid learning rate | Done here |
| Feature effective rank | Done here |
| Code-only control and the contribution gap | Done here. R(t) itself reported only if its t = 0 denominator is not near zero |
| Joint upper bound | Blocked. It needs one model over all 80 classes, and each task gets its own ten-output classifier |
| Adapter geometry | Not applicable. There is no adapter |
| DTD and Flowers102 probe tasks | Blocked. 47 and 102 classes cannot be generated |

Two pieces of work remain after that, neither of them a new metric:

- **Mid-stream probe points.** t = 2, 4 and 6 need the stream re-run with probing inline,
  because the study kept no mid-stream checkpoints.
- **Orders 02 and 03.** Without them, task identity and position stay confounded, so
  notebook 28's falling curve cannot be read.